In [8]:
!pip install yfinance tensorflow scikit-learn


In [9]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from datetime import datetime


In [10]:
symbol="NIFTY"
start = "2015-01-01"
end = datetime.today().strftime("%Y-%m-%d")

In [11]:
data = yf.download(symbol, start=start, end=end)

df = data.copy()

# Moving Averages
df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()

# Returns
df["Returns"] = df["Close"].pct_change()

# Volatility
df["Volatility"] = df["Returns"].rolling(20).std()

# RSI
delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))

# Direction Target
df["Tomorrow_Return"] = df["Close"].pct_change().shift(-1)
df["Direction"] = np.where(df["Tomorrow_Return"] > 0, 1, 0)

# Remove NaN
df = df.dropna()

df.head()


/tmp/ipykernel_596/419098896.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start, end=end)
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NIFTY"}}}
[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NIFTY']: YFTzMissingError('possibly delisted; no timezone found')


Price,Adj Close,Close,High,Low,Open,Volume,MA20,MA50,Returns,Volatility,RSI,Tomorrow_Return,Direction
Ticker,NIFTY,NIFTY,NIFTY,NIFTY,NIFTY,NIFTY,,,,,,,
Date,,,,,,,,,,,,,


In [12]:
features = [
    "Close",
    "MA20",
    "MA50",
    "RSI",
    "Volatility",
    "Volume"
]

X_data = df[features].values
y_data = df["Direction"].values


from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_data)




ValueError: Found array with 0 sample(s) (shape=(0, 6)) while a minimum of 1 is required by MinMaxScaler.

In [ ]:
X = []
y = []

window = 60

for i in range(window, len(X_scaled)):

    X.append(X_scaled[i-window:i])
    y.append(y_data[i])


X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)




In [ ]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]


In [ ]:
model = Sequential()

model.add(
    LSTM(64, return_sequences=True,
         input_shape=(60, len(features)))
)

model.add(Dropout(0.3))


model.add(
    LSTM(64)
)

model.add(Dropout(0.3))


model.add(Dense(1, activation="sigmoid"))


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()



In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)




In [ ]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", round(acc*100,2), "%")



In [ ]:
plt.figure(figsize=(12,6))

plt.plot(real, label="Actual")
plt.plot(predicted, label="Predicted")

plt.legend()
plt.title("NIFTY Prediction Using LSTM")

plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

# Confusion Matrix
y_pred = (model.predict(X_test) > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)


# Tomorrow Prediction
last_60 = X_scaled[-60:]

last_60 = last_60.reshape(1,60,len(features))

prob_up = model.predict(last_60)[0][0]

print("Probability of UP Tomorrow:", round(prob_up*100,2), "%")

if prob_up > 0.55:
    print("Signal: BUY")
elif prob_up < 0.45:
    print("Signal: SELL")
else:
    print("Signal: HOLD")



In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(real, predicted))
mae = mean_absolute_error(real, predicted)

print("RMSE:", round(rmse, 2))
print("MAE :", round(mae, 2))


In [ ]:
real_dir = np.sign(real[1:] - real[:-1])
pred_dir = np.sign(predicted[1:] - predicted[:-1])

direction_acc = np.mean(real_dir == pred_dir)

print("Direction Accuracy:", round(direction_acc*100, 2), "%")


In [ ]:
# Baseline: Tomorrow = Today
baseline = real[:-1]

baseline_rmse = np.sqrt(
    mean_squared_error(real[1:], baseline)
)

print("Baseline RMSE:", round(baseline_rmse, 2))


In [ ]:
model.save("nifty_lstm_model.keras")

print("Model Saved Successfully")
